# 01 — First bounds: from a table to a causal interval

**partialid** computes *numeric partial-identification bounds* for causal
queries in discrete data. You do not need a fully identified DAG. The main
idea is **assumption composition**: pick assumptions that appear in the causal
inference literature, combine them with `&`, and call `bound`.

This notebook is a complete first analysis. You will:

1. Inspect a small observational extract from a job-training study.
2. State a causal question in words and as a `Query`.
3. Compute Manski (worst-case confounding) bounds.
4. Read a `BoundResult` (`width`, `tight`, `method`, …).
5. Tighten the interval by adding monotone-response assumptions.

You do not need any other package docs to follow along.


## The study

We evaluate an observational **job-training program**. For each person we see:

- `treated` — whether they participated in the program (0/1)
- `employed` — whether they are employed at follow-up (0/1)

People select into training for reasons we do not fully observe (motivation,
local labor markets, …), so a simple comparison of employment rates between
treated and untreated units is **not** the causal effect of the program.

**Scientific question:** if *everyone* had been treated, what fraction would be
employed? That is the interventional probability
$P(\mathrm{employed}=1 \mid \mathrm{do}(\mathrm{treated}=1))$.


## Load and inspect the data


In [1]:
from pathlib import Path

def tutorial_data(name: str) -> Path:
    """Resolve notebooks/data/<name> whether __file__ is set (CI) or not."""
    here = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    candidates = [
        here / "data" / name,
        here / "notebooks" / "data" / name,
        Path.cwd() / "notebooks" / "data" / name,
        Path.cwd() / "data" / name,
    ]
    for p in candidates:
        if p.is_file():
            return p
    raise FileNotFoundError(f"could not find notebooks/data/{name}; tried {candidates}")

import pandas as pd

df = pd.read_csv(tutorial_data("job_training.csv"))
print(f"N = {len(df)} individuals")
print(df.head(8))
print()
print("treatment rate:", round(df["treated"].mean(), 3))
print("overall employment rate:", round(df["employed"].mean(), 3))
print()
print("Employment rate by treatment:")
print(df.groupby("treated")["employed"].mean().rename("P(employed=1 | treated)"))
print()
ct = pd.crosstab(df["treated"], df["employed"], margins=True)
print("Counts:")
print(ct)
print()
print("Row percentages (within treated):")
print(pd.crosstab(df["treated"], df["employed"], normalize="index").round(3))


N = 2000 individuals
   treated  employed
0        1         1
1        0         1
2        1         1
3        1         1
4        0         0
5        1         1
6        1         1
7        1         1

treatment rate: 0.496
overall employment rate: 0.558

Employment rate by treatment:
treated
0    0.404762
1    0.712702
Name: P(employed=1 | treated), dtype: float64

Counts:
employed    0     1   All
treated                  
0         600   408  1008
1         285   707   992
All       885  1115  2000

Row percentages (within treated):
employed      0      1
treated               
0         0.595  0.405
1         0.287  0.713


Among the treated, employment is high — but that **associational** success rate
mixes the program's effect with selection. Partial identification asks: given
only the observed table and stated assumptions, which values of the
*interventional* employment rate remain possible?


## Turn the table into partialid objects


In [2]:
from partialid import Assumptions, Query, bound, from_dataframe

data = from_dataframe(df, columns=["treated", "employed"])
print("variables:", data.variables.names)
print("domains:", data.variables.domains)
print("n_obs:", data.n_obs)
print("first probability atoms:")
for k, v in list(sorted(data.P.items()))[:4]:
    print(f"  P{k} = {v:.4f}")


variables: ('treated', 'employed')
domains: {'treated': (0, 1), 'employed': (0, 1)}
n_obs: 2000
first probability atoms:
  P(0, 0) = 0.3000
  P(0, 1) = 0.2040
  P(1, 0) = 0.1425
  P(1, 1) = 0.3535


`from_dataframe` builds an `ObservationalData` object: discrete `Variables`
plus an empirical joint law `P` (normalized cell probabilities). You will pass
`data.variables` and `data.P` to `bound`.


## State the causal question


In [3]:
Q = Query.interventional(outcome="employed", do={"treated": 1}, value=1)
print(Q.describe())

# Associational benchmark: employment among those who were actually treated
assoc = df.loc[df["treated"] == 1, "employed"].mean()
print(f"associational P(employed=1 | treated=1) = {assoc:.4f}")
print("This is *not* the same object as Q under confounding.")


P(employed=1 | do(treated=1))
associational P(employed=1 | treated=1) = 0.7127
This is *not* the same object as Q under confounding.


## Manski bounds (order-only / maximal confounding)

The Manski (natural) bounds keep only a causal **order**
`treated → employed` and allow arbitrary unobserved confounding. They are the
honest baseline when you are unwilling to restrict how selection works.


In [4]:
a_manski = Assumptions.manski("treated", "employed")
r = bound(data.variables, data.P, Q, a_manski)
print(r.summary(Q))


query: P(employed=1 | do(treated=1))
interval: [0.3535, 0.8575]
width: 0.504
method: fiber
tight: True
success: True


### How to read a `BoundResult`

| Field | Meaning |
|---|---|
| `lower`, `upper` | Endpoints of the identified interval |
| `width` | `upper - lower` (0 means point identification under these assumptions) |
| `success` | Whether the optimization succeeded / stayed feasible |
| `method` | Solver path chosen by `method="auto"` |
| `tight` | `True` if the interval is sharp for the stated assumptions; `False` means a valid **outer** approximation |

Always check `tight` before treating the interval as sharp. Leave
`method="auto"` unless you have a reason to override it.


In [5]:
print("interval:", r.interval)
print("width:", round(r.width, 4))
print("success:", r.success, "| method:", r.method, "| tight:", r.tight)
print("JSON-friendly:", r.to_dict()["interval"])


interval: (np.float64(0.3535), np.float64(0.8574999999999999))
width: 0.504
success: True | method: fiber | tight: True
JSON-friendly: [0.3535, 0.8574999999999999]


## Compose stronger assumptions with `&`

Adding assumptions **intersects** constraint sets: the feasible set can only
shrink, so the interval can only get narrower (or stay the same).

Here **MTR** (monotone treatment response) says employment is weakly increasing
in treatment for every latent type — “the program does not hurt anyone’s
employment prospects.” **MTS** (monotone treatment selection) restricts how
selection correlates with potential outcomes.


In [6]:
base = Assumptions.order("treated", "employed")
for label, a in [
    ("order only (Manski)", base),
    ("+ MTR", base & Assumptions.mtr("employed", "treated")),
    ("+ MTS", base & Assumptions.mts("treated", "employed")),
    ("+ MTR + MTS", base & Assumptions.mtr("employed", "treated") & Assumptions.mts("treated", "employed")),
]:
    rr = bound(data.variables, data.P, Q, a)
    print(f"{label:22s}  [{rr.lower:.4f}, {rr.upper:.4f}]  width={rr.width:.4f}")

print("\nAssumption list for MTR + MTS:")
for line in (base & Assumptions.mtr("employed", "treated") & Assumptions.mts("treated", "employed")).describe():
    print(" ", line)


order only (Manski)     [0.3535, 0.8575]  width=0.5040
+ MTR                   [0.5575, 0.8575]  width=0.3000
+ MTS                   [0.3535, 0.7127]  width=0.3592
+ MTR + MTS             [0.5575, 0.7127]  width=0.1552

Assumption list for MTR + MTS:
  OrderAssumption: OrderAssumption(order=('treated', 'employed'))
  MonotonicityAssumption: MonotonicityAssumption(outcome='employed', treatment='treated', direction='increasing')
  MTSAssumption: MTSAssumption(treatment='treated', outcome='employed', direction='increasing')


Under MTR the interval typically shrinks a lot relative to Manski: ruling out
“treatment hurts employment” removes many adversarial response types.


## Same table, different estimand: the ATE

Stakeholders often want the **average treatment effect**
\(P(Y_1=1) - P(Y_0=1)\) rather than the level under universal treatment.


In [7]:
Q_ate = Query.ate(outcome="employed", treatment="treated")
print(Q_ate.describe())
r_ate = bound(
    data.variables, data.P, Q_ate,
    Assumptions.manski("treated", "employed") & Assumptions.mtr("employed", "treated"),
)
print(r_ate.summary(Q_ate))


ATE: P(employed=1 | do(treated=1)) - P(employed=1 | do(treated=0))
query: ATE: P(employed=1 | do(treated=1)) - P(employed=1 | do(treated=0))
interval: [0, 0.6535]
width: 0.6535
method: fiber
tight: True
success: True


## You can now…

- Load discrete observational data and inspect it before bounding.
- Build a `Query` for an interventional probability or an ATE.
- Compute Manski bounds and tighten them by composing atoms with `&`.
- Read `width`, `tight`, `method`, and `success` on a `BoundResult`.

**Next:** [02_assumption_composition.ipynb](02_assumption_composition.ipynb) —
structural faces vs sensitivity budgets, and when to use recipes.
